# JAX + Flax Quantum Attention Baseline

This notebook demonstrates a minimal integration of a **multi-head attention**-style
module built with Flax and a **PennyLane FlaxLayer** to show compatibility with the
JAX/Flax ecosystem. It serves as a toy benchmark for the QMLHEP7 quantum particle
transformer project. The classical attention block is replaced by a quantum layer
on the output to highlight how to swap in quantum modules during model construction.

In [ ]:
# Install dependencies (run once)
!pip install -q jax jaxlib flax pennylane optax

In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import pennylane as qml

print("JAX version", jax.__version__)
print("Flax version", nn.__version__)
print("PennyLane version", qml.__version__)

## Define a simple quantum circuit and FlaxLayer

In [ ]:
# one-qubit circuit that rotates by an input and a trainable weight

dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev, interface="jax")
def qcircuit(inputs, weights):
    # inputs expected shape (1,) and weights shape (1,)
    qml.RX(inputs[0], wires=0)
    qml.RY(weights[0], wires=0)
    return qml.expval(qml.PauliZ(0))

weight_shapes = {"weights": (1,)}
qlayer = qml.qnn.FlaxLayer(qcircuit, weight_shapes)

# quick check
x = jnp.array([0.1])
w = jnp.array([0.2])
print("quantum output", qlayer(x, w))

## Multi-Head Attention Module with Quantum Output

In [ ]:
class QuantumAttentionBlock(nn.Module):
    dim: int
    num_heads: int = 1

    def setup(self):
        self.q_dense = nn.Dense(self.dim)
        self.k_dense = nn.Dense(self.dim)
        self.v_dense = nn.Dense(self.dim)
        self.out_dense = nn.Dense(self.dim)
        self.quantum = qlayer

    def __call__(self, x):
        q = self.q_dense(x)
        k = self.k_dense(x)
        v = self.v_dense(x)
        # simple scaled dot-product attention for demonstration
        score = jnp.einsum("...d,...d->...", q, k) / jnp.sqrt(self.dim)
        weights = nn.softmax(score)
        attended = weights[..., None] * v
        classical = self.out_dense(attended)
        # apply quantum layer to first feature vector in batch
        qout = self.quantum(classical[..., 0], jnp.ones((1,)))
        # broadcast quantum result and add to classical output
        return classical + qout[..., None]

# instantiate and run
model = QuantumAttentionBlock(dim=4)
params = model.init(jax.random.PRNGKey(0), jnp.ones((2,4)))
output = model.apply(params, jnp.ones((2,4)))
print("output shape", output.shape)

## Next Steps

- Extend to genuine multi-head implementation.
- Swap `QuantumAttentionBlock` into larger transformer models.
- Benchmark against classical-only baselines using jet classification data.

This notebook establishes the **basic pattern** for plugging a PennyLane `FlaxLayer`
into a JAX/Flax architecture. For QMLHEP7, similar bridges will be used inside a
particle transformer attention block.